# Experiment 2: GLCM + Random Forest

**Research Question:** How useful is texture information for classifying brain tumors?

We extract **Gray Level Co-occurrence Matrix (GLCM)** features to capture texture properties like contrast, homogeneity, energy, and correlation. 
We then train a **Random Forest** classifier on these texture features.

# Step 0: Google Colab Setup

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("\nGoogle Drive Mounted successfully!")
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")

# Step 1: Imports and Basics

In [ ]:
import os
import cv2
import numpy as np
from skimage.feature import graycomatrix, graycoprops
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

LABEL_MAP = {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
CLASS_NAMES = list(LABEL_MAP.keys())

# Step 2: Load Data & Extract GLCM Features
GLCM computes the frequency of pixel pairs with specific values and spatial relationships.
We extract several properties (contrast, dissimilarity, homogeneity, energy, correlation, ASM) from the GLCM to form our feature vector.

In [ ]:
def extract_glcm(image, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4]):
    if image.dtype != np.uint8:
        image = image.astype(np.uint8)
    
    glcm = graycomatrix(image, distances=distances, angles=angles, levels=256, symmetric=True, normed=True)
    
    properties = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    img_features = []
    for prop in properties:
        prop_vals = graycoprops(glcm, prop).flatten()
        img_features.extend(prop_vals)
    return np.array(img_features)

def load_and_extract_glcm(base_path, target_size=(128, 128)):
    X, y = [], []
    if not os.path.exists(base_path):
        print(f"ERROR: The path {base_path} does not exist! Please check your base_dir variable.")
        return np.array(X), np.array(y)

    print(f"Loading and extracting GLCM features from {base_path}...")
    for class_name, label_idx in tqdm(LABEL_MAP.items(), desc="Classes"):
        class_folder = os.path.join(base_path, class_name)
        if not os.path.exists(class_folder):
            continue

        for filename in os.listdir(class_folder):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(class_folder, filename)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img_resized = cv2.resize(img, target_size)
                    glcm_features = extract_glcm(img_resized)
                    X.append(glcm_features)
                    y.append(label_idx)
    return np.array(X), np.array(y)

base_dir = '/content/drive/MyDrive/NeuroScan'

# --- Auto-detect for local execution ---
if 'base_dir' not in locals():
    current_dir = os.getcwd()
    base_dir = current_dir if os.path.exists(os.path.join(current_dir, 'data', 'Training')) else os.path.dirname(current_dir)

train_dir = os.path.join(base_dir, 'data', 'Training')
test_dir  = os.path.join(base_dir, 'data', 'Testing')

print("Loading Training Data...")
X_train, y_train = load_and_extract_glcm(train_dir)

print("Loading Testing Data...")
X_test, y_test = load_and_extract_glcm(test_dir)

if len(X_train) > 0:
    print(f"\nTraining GLCM feature shape: {X_train.shape}")
    print(f"Testing GLCM feature shape:  {X_test.shape}")
else:
    print("\nFailed to load images. Please fix the base_dir path above.")

# Step 3: Train the Random Forest

In [ ]:
print("Training Random Forest on GLCM features...")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Training Complete!")

# Step 4: Evaluation

In [ ]:
y_pred = model.predict(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')

print("================ EVALUATION METRICS ================")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print("====================================================")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix: GLCM using Random Forest')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()